# Breast-VLReport — BUS-BRA (development) → BrEaST (locked external test)

Run top to bottom on a **GPU runtime** (Runtime → Change runtime type → T4 GPU).

**Read before trusting results**
- Neither dataset has narrative radiology reports. Reports are **templates** built from BI-RADS, pathology and mask-derived geometry, so BLEU/CIDEr measure template fidelity only.
- This pipeline was smoke-tested on synthetic data only. On the first real run, check the column/file matching printed in step 5.
- Not implemented: multi-view fusion, clinical-consistency loss, CLIP-GPT/BLIP-2 baselines, explainability maps, paired statistical tests.

## 1. Setup

In [ ]:
import torch, subprocess
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-- switch to a GPU runtime!")
!pip -q install timm pycocoevalcap rouge-score scikit-image openpyxl nltk bert-score transformers
import nltk; nltk.download("wordnet", quiet=True); nltk.download("omw-1.4", quiet=True)

## 2. Source files (written to the Colab disk)

In [ ]:
%%writefile data.py
"""Datasets, structured-attribute extraction, template reports, tokenizer, patient-level splits.

IMPORTANT: neither BUS-BRA nor BrEaST ships narrative radiology reports. Reports here are TEMPLATE
reports built from structured annotations (BI-RADS, pathology, mask-derived geometry and, for BrEaST,
expert BI-RADS descriptors). Text metrics therefore measure fidelity to the template, not radiologist style.
"""
import re, random, glob, os
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import StratifiedGroupKFold

IMG = 224
MEAN, STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

# ---- attribute <-> report slot definitions -------------------------------------------------------------
SLOT_KEY = {  # attribute name -> phrase used in the report ("<key> is <phrase>.")
    "size": "relative size", "outline": "outline", "orientation": "orientation", "shape": "shape",
    "margin": "margin", "echogenicity": "echogenicity", "posterior": "posterior feature",
    "pathology": "assessment", "birads": "birads category"}
PHRASE = {  # value -> phrase (kept distinct from each other inside one attribute)
    "pathology": {"benign": "most consistent with a benign lesion", "malignant": "suspicious for malignancy"},
    "orientation": {"wide": "wider than tall", "tall": "taller than wide"},
}
ORDER_FIND = ["size", "outline", "orientation", "shape", "margin", "echogenicity", "posterior"]
ORDER_IMP = ["pathology", "birads"]
# mask-derived geometric proxies (thresholds are arbitrary constants -> document/tune them)
SIZE_T = (0.02, 0.08)           # fraction of image area: small < 0.02 <= medium < 0.08 <= large
SOLIDITY_T, CIRC_T = 0.90, 0.60  # outline "irregular" if solidity<0.90 or circularity<0.60


def phrase(attr, v):
    return PHRASE.get(attr, {}).get(v, v)


def build_report(attrs: dict) -> str:
    f = [f"{SLOT_KEY[a]} is {phrase(a, attrs[a])}." for a in ORDER_FIND if attrs.get(a)]
    i = [f"{SLOT_KEY[a]} is {phrase(a, attrs[a])}." for a in ORDER_IMP if attrs.get(a)]
    return "findings: a lesion is seen. " + " ".join(f) + " impression: " + " ".join(i)


def norm_text(t: str) -> str:
    t = t.lower()
    t = re.sub(r"\s+([.,;:])", r"\1", t)
    return re.sub(r"\s+", " ", t).strip()


def parse_report(text: str, attr_names) -> dict:
    """Read stated facts back out of a (generated) report. Returns {attr: value | '<invalid>'}."""
    text, out = norm_text(text), {}
    for a in attr_names:
        m = re.search(rf"{re.escape(SLOT_KEY[a])} is ([^.]+)\.", text)
        if m:
            inv = {phrase(a, v): v for v in ALL_VALUES.get(a, [])}
            out[a] = inv.get(m.group(1).strip(), "<invalid>")
    return out


ALL_VALUES = {}  # filled by register_values(); used by the parser to map phrases back to values
def register_values(schema):
    for a, vals in schema.items():
        ALL_VALUES[a] = list(vals)


# ---- mask geometry -------------------------------------------------------------------------------------
def mask_attrs(mask: np.ndarray) -> dict:
    from skimage.measure import label, regionprops
    lab = label(mask > 0)
    if lab.max() == 0:
        return {}
    r = max(regionprops(lab), key=lambda r: r.area)
    frac = r.area / mask.size
    circ = 4 * np.pi * r.area / (r.perimeter ** 2 + 1e-6)
    h, w = r.bbox[2] - r.bbox[0], r.bbox[3] - r.bbox[1]
    return {"size": "small" if frac < SIZE_T[0] else "medium" if frac < SIZE_T[1] else "large",
            "outline": "irregular" if (r.solidity < SOLIDITY_T or circ < CIRC_T) else "regular",
            "orientation": "wide" if w >= h else "tall"}


def union_mask(paths):
    m = None
    for p in paths:
        a = np.array(Image.open(p).convert("L")) > 0
        m = a if m is None else (m | a)
    return m.astype(np.uint8)


# ---- dataset adapters ----------------------------------------------------------------------------------
def _find(df, *keys):
    cols = {re.sub(r"[^a-z0-9]", "", c.lower()): c for c in df.columns}
    for k in keys:
        for n, c in cols.items():
            if k in n:
                return c
    return None


def load_busbra(root) -> pd.DataFrame:
    root = Path(root)
    csv = next(iter(root.rglob("bus_data.csv")), None) or next(iter(root.rglob("*.csv")))
    df = pd.read_csv(csv)
    print("[BUS-BRA] columns:", list(df.columns))
    c_id, c_bi, c_pa = _find(df, "id"), _find(df, "birads"), _find(df, "pathology")
    c_case, c_dev = _find(df, "case"), _find(df, "device", "scanner")
    imgs = {p.stem: p for p in root.rglob("*.png")}
    rows = []
    for _, r in df.iterrows():
        sid = str(r[c_id])
        ip = imgs.get(sid)
        mp = imgs.get(sid.replace("bus_", "mask_")) or imgs.get("mask_" + sid.split("_", 1)[-1])
        if ip is None or mp is None:
            continue
        m = np.array(Image.open(mp).convert("L"))
        a = mask_attrs(m)
        if not a:
            continue
        a["birads"] = str(r[c_bi])[0]
        a["pathology"] = str(r[c_pa]).strip().lower()
        grp = str(r[c_case]) if c_case else re.sub(r"\D", "", sid.split("_")[-1].split("-")[0])
        rows.append(dict(sample=sid, img=str(ip), mask=str(mp), group=f"busbra_{grp}",
                         device=str(r[c_dev]) if c_dev else "NA", att=a, source="busbra"))
    assert rows, "No BUS-BRA samples matched. Check file names vs. CSV IDs."
    return pd.DataFrame(rows)


def load_breast(root, expert=False) -> pd.DataFrame:
    root = Path(root)
    xl = next(iter(root.rglob("*.xlsx")))
    df = pd.read_excel(xl)
    print("[BrEaST] columns:", list(df.columns))
    c_f, c_bi, c_cl = _find(df, "filename", "image"), _find(df, "birads"), _find(df, "classification")
    c_id = _find(df, "caseid", "case")
    ex = {"shape": _find(df, "shape"), "margin": _find(df, "margin"),
          "echogenicity": _find(df, "echogenicity"), "posterior": _find(df, "posterior")}
    files = {p.name: p for p in root.rglob("*.png")}
    rows = []
    for _, r in df.iterrows():
        cls = str(r[c_cl]).lower()
        if "normal" in cls or not (("benign" in cls) or ("malignant" in cls)):
            continue
        fn = str(r[c_f])
        ip = files.get(fn)
        stem = Path(fn).stem
        mps = [p for n, p in files.items() if n.startswith(stem + "_tumor")]
        if ip is None or not mps:
            continue
        a = mask_attrs(union_mask(mps))
        if not a:
            continue
        a["birads"] = re.sub(r"\D", "", str(r[c_bi]))[:1] or "NA"
        a["pathology"] = "malignant" if "malignant" in cls else "benign"
        if expert:
            for k, c in ex.items():
                v = str(r[c]).strip().lower() if c else ""
                if v and v != "nan":
                    a[k] = "not circumscribed" if (k == "margin" and not v.startswith("circ")) else v
        rows.append(dict(sample=stem, img=str(ip), mask=str(mps[0]), mask_all=[str(p) for p in mps],
                         group=f"breast_{r[c_id] if c_id else stem}", device="BrEaST", att=a, source="breast"))
    assert rows, "No BrEaST samples matched. Print the columns above and adjust _find() keys."
    return pd.DataFrame(rows)


def load_dataset(name, root, expert=False):
    df = load_busbra(root) if name == "busbra" else load_breast(root, expert)
    return df[df.att.map(lambda a: a.get("birads", "NA") in "2345")].reset_index(drop=True)


def make_split(df, folds=10, fold=0, seed=0):
    """Patient-level split: test=fold, val=fold+1, train=rest; stratified by pathology; asserts no leakage."""
    y = df.att.map(lambda a: a["pathology"]).values
    sg = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed)
    parts = [idx for _, idx in sg.split(df, y, df.group)]
    te, va = parts[fold % folds], parts[(fold + 1) % folds]
    tr = np.setdiff1d(np.arange(len(df)), np.concatenate([te, va]))
    g = lambda i: set(df.group.iloc[i])
    assert not (g(tr) & g(va)) and not (g(tr) & g(te)) and not (g(va) & g(te)), "patient leakage!"
    return df.iloc[tr].reset_index(drop=True), df.iloc[va].reset_index(drop=True), df.iloc[te].reset_index(drop=True)


def build_schema(df):
    s = {}
    for a in df.att:
        for k, v in a.items():
            s.setdefault(k, set()).add(v)
    return {k: sorted(v) for k, v in s.items()}


# ---- tokenizer -----------------------------------------------------------------------------------------
class Tok:
    def __init__(self, texts=None, hf=None, max_len=96):
        self.max_len, self.hf = max_len, None
        if hf:
            from transformers import AutoTokenizer
            self.hf = AutoTokenizer.from_pretrained(hf)
            self.pad = self.eos = self.bos = self.hf.eos_token_id
            self.V = len(self.hf)
        else:
            self.pad, self.bos, self.eos, self.unk = 0, 1, 2, 3
            words = sorted({w for t in texts for w in self.split(t)})
            self.itos = ["<pad>", "<bos>", "<eos>", "<unk>"] + words
            self.stoi = {w: i for i, w in enumerate(self.itos)}
            self.V = len(self.itos)

    @staticmethod
    def split(t):
        return re.findall(r"[a-z0-9]+(?:-[a-z0-9]+)*|[.,;:]", t.lower())

    def encode(self, t):
        ids = self.hf.encode(norm_text(t)) if self.hf else [self.stoi.get(w, self.unk) for w in self.split(t)]
        ids = [self.bos] + ids[: self.max_len - 2] + [self.eos]
        return ids + [self.pad] * (self.max_len - len(ids))

    def decode(self, ids):
        ids = [int(i) for i in ids]
        if self.hf:
            cut = ids.index(self.eos, 1) if self.eos in ids[1:] else len(ids)
            return norm_text(self.hf.decode(ids[1:cut]))
        out = []
        for i in ids:
            if i == self.eos: break
            if i > 3: out.append(self.itos[i])
        return norm_text(" ".join(out))


# ---- torch dataset -------------------------------------------------------------------------------------
def letterbox(im, resample):
    w, h = im.size
    s = max(w, h)
    bg = Image.new(im.mode, (s, s), 0)
    bg.paste(im, ((s - w) // 2, (s - h) // 2))
    return bg.resize((IMG, IMG), resample)


class BUSData(Dataset):
    def __init__(self, df, schema, tok, train=False):
        self.df, self.schema, self.tok, self.train = df, schema, tok, train
        self.reports = [build_report(a) for a in df.att]

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        im = letterbox(Image.open(r["img"]).convert("RGB"), Image.BILINEAR)
        mk = letterbox(Image.fromarray(union_mask(r.mask_all if isinstance(r.get("mask_all"), list) else [r["mask"]]) * 255),
                       Image.NEAREST)
        x = np.asarray(im, dtype=np.float32) / 255.0
        m = (np.asarray(mk) > 127).astype(np.float32)
        if self.train:
            if random.random() < 0.5: x, m = x[:, ::-1], m[:, ::-1]
            x = np.clip((x - 0.5) * random.uniform(0.85, 1.15) + 0.5 + random.uniform(-0.08, 0.08), 0, 1)
        x = torch.from_numpy(((x - MEAN) / STD).transpose(2, 0, 1).copy()).float()
        lab = {a: (self.schema[a].index(r.att[a]) if a in r.att and r.att[a] in self.schema[a] else -100)
               for a in self.schema}
        return dict(img=x, mask=torch.from_numpy(m.copy())[None], ids=torch.tensor(self.tok.encode(self.reports[i])),
                    labels=torch.tensor([lab[a] for a in self.schema]), idx=i)

In [ ]:
%%writefile model.py
"""Breast-VLReport and baselines. One ReportModel class; flags switch lesion / attribute / contrastive parts,
so baselines and ablations (G: direct image->report, H: +lesion, I: +attributes, A: full) share one code path."""
import math, random
import torch, torch.nn as nn, torch.nn.functional as F
import timm

ENCODERS = {"swin": "swin_tiny_patch4_window7_224", "vit": "vit_base_patch16_224", "cnn": "resnet50"}


class Encoder(nn.Module):
    def __init__(self, kind, pretrained=True):
        super().__init__()
        self.m = timm.create_model(ENCODERS[kind], pretrained=pretrained, num_classes=0, global_pool="")
        self.dim = self.m.num_features

    def forward(self, x):
        f = self.m.forward_features(x)
        if f.dim() == 4:  # NHWC (swin) or NCHW (resnet)
            if f.shape[-1] == self.dim and f.shape[1] != self.dim:
                f = f.permute(0, 3, 1, 2)
            B, C, H, W = f.shape
            return f.flatten(2).transpose(1, 2), (H, W)
        N = f.shape[1]; g = math.isqrt(N)  # ViT: drop cls/prefix tokens
        return f[:, N - g * g:], (g, g)


class SegHead(nn.Module):
    def __init__(self, c):
        super().__init__()
        blk = lambda i, o: nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True))
        self.net = nn.Sequential(blk(c, 128), nn.Upsample(scale_factor=2), blk(128, 64), nn.Upsample(scale_factor=2),
                                 blk(64, 32), nn.Upsample(scale_factor=2), nn.Conv2d(32, 1, 1))

    def forward(self, fmap):
        return F.interpolate(self.net(fmap), size=(224, 224), mode="bilinear", align_corners=False)


# ---- decoders: forward(ids, memory) -> logits; embed(ids) -> token embeddings ------------------------------
class TfmDecoder(nn.Module):
    def __init__(self, V, d, layers=4, heads=8, max_len=128):
        super().__init__()
        self.emb, self.pos = nn.Embedding(V, d), nn.Parameter(torch.zeros(max_len, d))
        self.dec = nn.TransformerDecoder(nn.TransformerDecoderLayer(d, heads, 4 * d, 0.1, batch_first=True, norm_first=True), layers)
        self.norm, self.out = nn.LayerNorm(d), nn.Linear(d, V)

    def embed(self, ids): return self.emb(ids)

    def forward(self, ids, mem):
        L = ids.size(1)
        mask = torch.triu(torch.full((L, L), float("-inf"), device=ids.device), 1)
        return self.out(self.norm(self.dec(self.emb(ids) + self.pos[:L], mem, tgt_mask=mask)))


class LSTMDecoder(nn.Module):  # classical CNN-RNN baseline with dot-product attention over visual tokens
    def __init__(self, V, d):
        super().__init__()
        self.emb, self.cell = nn.Embedding(V, d), nn.LSTMCell(2 * d, d)
        self.init_h, self.out = nn.Linear(d, 2 * d), nn.Linear(2 * d, V)

    def embed(self, ids): return self.emb(ids)

    def forward(self, ids, mem):
        h, c = self.init_h(mem.mean(1)).chunk(2, -1)
        outs = []
        for t in range(ids.size(1)):
            a = torch.softmax((mem @ h.unsqueeze(-1)).squeeze(-1) / math.sqrt(h.size(-1)), -1)
            ctx = (a.unsqueeze(-1) * mem).sum(1)
            h, c = self.cell(torch.cat([self.emb(ids[:, t]), ctx], -1), (h, c))
            outs.append(self.out(torch.cat([h, ctx], -1)))
        return torch.stack(outs, 1)


class GPT2Decoder(nn.Module):  # pretrained GPT-2 + (newly initialised) cross-attention to visual/clinical tokens
    def __init__(self, name="gpt2"):
        super().__init__()
        from transformers import GPT2Config, GPT2LMHeadModel
        cfg = GPT2Config.from_pretrained(name); cfg.add_cross_attention = True
        self.m, self.d = GPT2LMHeadModel.from_pretrained(name, config=cfg), cfg.n_embd

    def embed(self, ids): return self.m.transformer.wte(ids)
    def forward(self, ids, mem): return self.m(input_ids=ids, encoder_hidden_states=mem).logits


class ReportModel(nn.Module):
    def __init__(self, tok, schema, enc="swin", dec="tfm", lesion=True, attr=True, contrastive=True,
                 d=512, pretrained=True, gpt2="gpt2"):
        super().__init__()
        self.tok, self.schema, self.names = tok, schema, list(schema)
        self.use_lesion, self.use_attr, self.use_con = lesion, attr, contrastive
        self.enc = Encoder(enc, pretrained)
        if dec == "gpt2":
            self.dec = GPT2Decoder(gpt2); d = self.dec.d
        else:
            self.dec = LSTMDecoder(tok.V, d) if dec == "lstm" else TfmDecoder(tok.V, d)
        self.d = d
        self.proj = nn.Linear(self.enc.dim, d)
        if lesion:
            self.seg = SegHead(self.enc.dim)
            self.lesion_embed = nn.Parameter(torch.zeros(d))
            self.extra = nn.Parameter(torch.zeros(2, d))  # type embeddings for [global, lesion] tokens
        if attr:
            assert lesion, "attribute module reads the lesion vector"
            self.heads = nn.ModuleDict({a: nn.Linear(2 * d, len(v)) for a, v in schema.items()})
            self.attr_emb = nn.ModuleDict({a: nn.Linear(len(v), d, bias=False) for a, v in schema.items()})
        if contrastive:
            self.pi, self.pt, self.logit_scale = nn.Linear(d, 256), nn.Linear(d, 256), nn.Parameter(torch.tensor(2.6))

    def encode(self, img, gt_mask=None, p_gt=0.5):
        tok, (h, w) = self.enc(img)
        B, N, _ = tok.shape
        mem, out = self.proj(tok), {}
        if not self.use_lesion:
            return mem, out, mem.mean(1)
        fmap = tok.transpose(1, 2).reshape(B, -1, h, w)
        out["seg"] = self.seg(fmap)
        m = torch.sigmoid(out["seg"])
        if self.training and gt_mask is not None and random.random() < p_gt:
            m = gt_mask
        mt = F.adaptive_avg_pool2d(m, (h, w)).flatten(1).unsqueeze(-1)  # B,N,1
        lesion = (mt * mem).sum(1) / (mt.sum(1) + 1e-6)
        glob = mem.mean(1)
        mem = mem + mt * self.lesion_embed
        extra = [glob + self.extra[0], lesion + self.extra[1]]
        if self.use_attr:
            z = torch.cat([glob, lesion], -1)
            out["attr"] = {a: self.heads[a](z) for a in self.names}
            extra += [self.attr_emb[a](F.softmax(out["attr"][a], -1)) for a in self.names]
        return torch.cat([torch.stack(extra, 1), mem], 1), out, lesion

    def forward(self, batch):
        mem, out, vis = self.encode(batch["img"], batch.get("mask"))
        out["logits"] = self.dec(batch["ids"][:, :-1], mem)
        if self.use_con:
            ti = F.normalize(self.pt(self.dec.embed(batch["ids"]).mean(1)), dim=-1)
            vi = F.normalize(self.pi(vis), dim=-1)
            out["sim"] = self.logit_scale.exp().clamp(max=100) * vi @ ti.t()
        return out

    @torch.no_grad()
    def generate(self, img, max_len=None):
        self.eval()
        mem, out, _ = self.encode(img)
        B, L = img.size(0), max_len or self.tok.max_len
        ids = torch.full((B, 1), self.tok.bos, device=img.device)
        done = torch.zeros(B, dtype=torch.bool, device=img.device)
        for _ in range(L - 1):
            nxt = self.dec(ids, mem)[:, -1].argmax(-1)
            nxt = torch.where(done, torch.full_like(nxt, self.tok.pad), nxt)
            ids = torch.cat([ids, nxt[:, None]], 1)
            done |= nxt == self.tok.eos
            if done.all(): break
        return ids, out


def compute_losses(model, batch, out):
    """Returns dict of the individual losses (weights applied by the trainer)."""
    ids = batch["ids"]
    tgt, eos = ids[:, 1:], model.tok.eos
    seen_eos = (tgt == eos).cumsum(1) - (tgt == eos).long()   # positions up to and including first eos are valid
    valid = seen_eos == 0
    L = {"ce": F.cross_entropy(out["logits"][valid], tgt[valid])}
    if "seg" in out:
        p, g = torch.sigmoid(out["seg"]), batch["mask"]
        dice = 1 - (2 * (p * g).sum((1, 2, 3)) + 1) / (p.sum((1, 2, 3)) + g.sum((1, 2, 3)) + 1)
        L["seg"] = dice.mean() + F.binary_cross_entropy_with_logits(out["seg"], g)
    if "attr" in out:
        ls = [F.cross_entropy(out["attr"][a], batch["labels"][:, i], ignore_index=-100)
              for i, a in enumerate(model.names) if (batch["labels"][:, i] != -100).any()]
        L["attr"] = torch.stack(ls).mean()
    if "sim" in out:  # multi-positive InfoNCE: identical reports count as positives (templated text has many)
        pos = (ids[:, None, :] == ids[None, :, :]).all(-1).float()
        for s in (out["sim"], out["sim"].t()):
            lp = s.log_softmax(1)
            L["con"] = L.get("con", 0) + 0.5 * -((lp * pos).sum(1) / pos.sum(1)).mean()
    return L

In [ ]:
%%writefile train.py
"""Staged training. Proposed model: S1 visual pretraining (seg+attrs) -> S2 image-text alignment (contrastive)
-> S3 report CE -> S4 joint. Baselines: single CE stage.  NOTE: the clinical-consistency loss (ablation F) is NOT implemented."""
import argparse, json, random, time
from pathlib import Path
import numpy as np, torch
from torch.utils.data import DataLoader
import data as D
from model import ReportModel, compute_losses

p = argparse.ArgumentParser()
p.add_argument("--dev", default="busbra", choices=["busbra", "breast"]); p.add_argument("--dev_root", required=True)
p.add_argument("--expert_attrs", action="store_true", help="use BrEaST expert descriptors (option B only)")
p.add_argument("--fold", type=int, default=0); p.add_argument("--folds", type=int, default=10)
p.add_argument("--model", default="breast_vlreport", choices=["breast_vlreport", "cnn_lstm", "cnn_transformer", "vit_transformer", "swin_transformer"])
p.add_argument("--no_lesion", action="store_true"); p.add_argument("--no_attr", action="store_true"); p.add_argument("--no_contrastive", action="store_true")
p.add_argument("--decoder", default=None, choices=["tfm", "gpt2", "lstm"]); p.add_argument("--hf_tok", default=None)
p.add_argument("--epochs", type=int, nargs=4, default=[15, 5, 15, 15], help="S1 S2 S3 S4 (baselines use S3 only)")
p.add_argument("--bs", type=int, default=16); p.add_argument("--lr", type=float, default=3e-4); p.add_argument("--enc_lr", type=float, default=5e-5)
p.add_argument("--no_pretrained", action="store_true"); p.add_argument("--seed", type=int, default=0)
p.add_argument("--out", required=True); p.add_argument("--workers", type=int, default=2); p.add_argument("--limit", type=int, default=0)
a = p.parse_args()

random.seed(a.seed); np.random.seed(a.seed); torch.manual_seed(a.seed)
dev = "cuda" if torch.cuda.is_available() else "cpu"
out = Path(a.out); out.mkdir(parents=True, exist_ok=True)

df = D.load_dataset(a.dev, a.dev_root, a.expert_attrs)
if a.limit: df = df.sample(a.limit, random_state=0).reset_index(drop=True)
tr, va, te = D.make_split(df, a.folds, a.fold, a.seed)
for n, s in zip(["train", "val", "test"], [tr, va, te]):
    s.to_pickle(out / f"split_{n}.pkl")
schema = D.build_schema(tr); D.register_values(schema)
tok = D.Tok(texts=[D.build_report(x) for x in tr.att], hf=a.hf_tok)
print(f"train/val/test = {len(tr)}/{len(va)}/{len(te)} patients-disjoint | schema: {schema}")

enc = {"breast_vlreport": "swin", "swin_transformer": "swin", "vit_transformer": "vit"}.get(a.model, "cnn")
full = a.model == "breast_vlreport"
dec = a.decoder or ("lstm" if a.model == "cnn_lstm" else "tfm")
model = ReportModel(tok, schema, enc, dec, lesion=full and not a.no_lesion,
                    attr=full and not a.no_lesion and not a.no_attr,
                    contrastive=full and not a.no_contrastive, pretrained=not a.no_pretrained).to(dev)

mk = lambda s, t: DataLoader(D.BUSData(s, schema, tok, t), a.bs, shuffle=t, num_workers=a.workers, drop_last=t)
dl_tr, dl_va = mk(tr, True), mk(va, False)
move = lambda b: {k: (v.to(dev) if torch.is_tensor(v) else v) for k, v in b.items()}

# stage = (name, epochs, loss weights, freeze encoder)
if full:
    W = lambda **k: {"seg": 0, "attr": 0, "con": 0, "ce": 0, **k}
    stages = [("S1_visual", a.epochs[0], W(seg=1, attr=1), False), ("S2_align", a.epochs[1], W(seg=.5, attr=.5, con=1), False),
              ("S3_report", a.epochs[2], W(ce=1), True), ("S4_joint", a.epochs[3], W(ce=1, seg=.3, attr=.3, con=.1), False)]
    stages = [s for s in stages if s[1] > 0]
else:
    stages = [("CE", a.epochs[2], {"ce": 1}, False)]

log = []
for name, ep, w, freeze in stages:
    w = {k: v for k, v in w.items() if v}
    for q in model.enc.parameters(): q.requires_grad = not freeze
    enc_p = [q for q in model.enc.parameters() if q.requires_grad]
    ids_e = {id(q) for q in model.enc.parameters()}
    rest = [q for q in model.parameters() if id(q) not in ids_e and q.requires_grad]
    opt = torch.optim.AdamW([{"params": enc_p, "lr": a.enc_lr}, {"params": rest, "lr": a.lr}], weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, [a.enc_lr, a.lr], total_steps=max(1, ep * len(dl_tr)), pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=dev == "cuda")
    best, best_sd = 1e9, None

    def step(b, train):
        with torch.autocast(dev, dtype=torch.float16, enabled=dev == "cuda"):
            o = model(b); L = compute_losses(model, b, o)
            tot = sum(w[k] * L[k] for k in w if k in L)
        return tot, L

    for e in range(ep):
        model.train(); t0 = time.time()
        for b in dl_tr:
            tot, _ = step(move(b), True)
            opt.zero_grad(); scaler.scale(tot).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
        model.eval(); vt, n = 0, 0
        with torch.no_grad():
            for b in dl_va:
                t, _ = step(move(b), False); vt += t.item() * len(b["idx"]); n += len(b["idx"])
        vt /= max(n, 1)
        print(f"[{name}] epoch {e+1}/{ep} train_last={tot.item():.3f} val={vt:.3f} ({time.time()-t0:.0f}s)")
        log.append(dict(stage=name, epoch=e + 1, val=vt))
        if vt < best: best, best_sd = vt, {k: v.detach().clone() for k, v in model.state_dict().items()}
    if best_sd: model.load_state_dict(best_sd)   # model selection on VALIDATION only

torch.save({"state": model.state_dict(), "args": vars(a), "schema": schema,
            "tok": {"hf": a.hf_tok, "texts": [D.build_report(x) for x in tr.att], "max_len": tok.max_len}}, out / "model.pt")
json.dump(log, open(out / "log.json", "w"))
print("saved", out / "model.pt")

In [ ]:
%%writefile evaluate.py
"""Evaluate a trained model on val / internal test / locked external test and report the proposal's metrics.
The external set is only ever *evaluated*; the decision threshold comes from the VALIDATION split of the dev site."""
import argparse, json
from pathlib import Path
import numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score
import data as D
from model import ReportModel

p = argparse.ArgumentParser()
p.add_argument("--run", required=True); p.add_argument("--ext", choices=["busbra", "breast"]); p.add_argument("--ext_root")
p.add_argument("--bs", type=int, default=32); p.add_argument("--boot", type=int, default=1000)
a = p.parse_args()
dev = "cuda" if torch.cuda.is_available() else "cpu"
run = Path(a.run)
ck = torch.load(run / "model.pt", map_location="cpu", weights_only=False)
ar, schema = ck["args"], ck["schema"]; D.register_values(schema)
tok = D.Tok(texts=ck["tok"]["texts"], hf=ck["tok"]["hf"], max_len=ck["tok"]["max_len"])
full = ar["model"] == "breast_vlreport"
enc = {"breast_vlreport": "swin", "swin_transformer": "swin", "vit_transformer": "vit"}.get(ar["model"], "cnn")
dec = ar["decoder"] or ("lstm" if ar["model"] == "cnn_lstm" else "tfm")
model = ReportModel(tok, schema, enc, dec, lesion=full and not ar["no_lesion"], attr=full and not ar["no_lesion"] and not ar["no_attr"],
                    contrastive=full and not ar["no_contrastive"], pretrained=False)
model.load_state_dict(ck["state"]); model.to(dev).eval()
rng = np.random.default_rng(0)


@torch.no_grad()
def predict(df):
    dl = DataLoader(D.BUSData(df, schema, tok), a.bs, num_workers=2)
    texts, dice, attr, prob = [], [], {k: [] for k in schema}, []
    pi = schema["pathology"].index("malignant") if "malignant" in schema.get("pathology", []) else None
    for b in dl:
        ids, o = model.generate(b["img"].to(dev))
        texts += [tok.decode(r) for r in ids.cpu()]
        if "seg" in o:
            pr, g = (torch.sigmoid(o["seg"]) > .5).float().cpu(), b["mask"]
            dice += ((2 * (pr * g).sum((1, 2, 3)) + 1e-6) / (pr.sum((1, 2, 3)) + g.sum((1, 2, 3)) + 1e-6)).tolist()
        if "attr" in o:
            for k in schema: attr[k] += o["attr"][k].argmax(-1).cpu().tolist()
            prob += torch.softmax(o["attr"]["pathology"], -1)[:, pi].cpu().tolist()
    return dict(texts=texts, dice=np.array(dice), attr=attr, prob=np.array(prob))


def ci(x, fn):
    x = np.asarray(x); v = [fn(x[rng.integers(0, len(x), len(x))]) for _ in range(a.boot)]
    return [float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))]


def clinical(texts, df):
    """Facts stated in the generated report vs. the structured ground truth (template-level, NOT radiologist-level)."""
    names = [k for k in schema]
    stated = wrong = gt_total = missing = bad_reports = 0; per = []
    for t, at in zip(texts, df.att):
        pr = D.parse_report(t, names)
        s = w = 0
        for k, v in pr.items():
            if k in at: s += 1; w += int(v != at[k])        # only facts verifiable against GT
        gt_total += len([k for k in at if k in names]); missing += len([k for k in at if k in names and k not in pr])
        stated += s; wrong += w; bad_reports += int(w > 0); per.append((s, w))
    per = np.array(per)
    return dict(factual_error_rate=wrong / max(stated, 1), hallucination_report_rate=bad_reports / len(texts),
                omission_rate=missing / max(gt_total, 1),
                factual_error_rate_ci=ci(np.arange(len(per)), lambda i: per[i, 1].sum() / max(per[i, 0].sum(), 1))), per


def nlp(texts, df):
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.cider.cider import Cider
    gts = {i: [D.norm_text(D.build_report(x))] for i, x in enumerate(df.att)}
    res = {i: [t if t else "empty"] for i, t in enumerate(texts)}
    r = {}
    b, _ = Bleu(4).compute_score(gts, res, verbose=0); r.update({f"BLEU-{i+1}": float(v) for i, v in enumerate(b)})
    r["ROUGE-L"] = float(Rouge().compute_score(gts, res)[0]); r["CIDEr"] = float(Cider().compute_score(gts, res)[0])
    try:
        from nltk.translate.meteor_score import meteor_score
        r["METEOR"] = float(np.mean([meteor_score([gts[i][0].split()], res[i][0].split()) for i in gts]))
    except Exception as e: r["METEOR"] = f"unavailable ({type(e).__name__}; run nltk.download('wordnet'))"
    try:
        from bert_score import score
        r["BERTScore-F1"] = float(score([res[i][0] for i in gts], [gts[i][0] for i in gts], lang="en")[2].mean())
    except Exception: r["BERTScore-F1"] = "unavailable (pip install bert-score)"
    return r


def image_level(P, df, thr):
    r = {}
    if len(P["dice"]): r["Dice"] = float(P["dice"].mean()); r["Dice_ci"] = ci(P["dice"], np.mean)
    if len(P["prob"]):
        y = np.array([x["pathology"] == "malignant" for x in df.att]).astype(int)
        if 0 < y.sum() < len(y):
            r["AUROC"] = float(roc_auc_score(y, P["prob"]))
            idx = np.arange(len(y)); r["AUROC_ci"] = ci(idx, lambda i: roc_auc_score(y[i], P["prob"][i]) if 0 < y[i].sum() < len(i) else np.nan)
            pred = P["prob"] >= thr
            r["sensitivity"], r["specificity"] = float((pred & (y == 1)).sum() / y.sum()), float((~pred & (y == 0)).sum() / (y == 0).sum())
        for k in ("birads", "pathology", "outline", "size", "orientation"):
            if k in schema:
                gt = np.array([schema[k].index(x[k]) if x.get(k) in schema[k] else -1 for x in df.att])
                r[f"{k}_acc"] = float((np.array(P["attr"][k]) == gt).mean())
    return r


def youden(P, df):
    if not len(P["prob"]): return 0.5
    y = np.array([x["pathology"] == "malignant" for x in df.att]).astype(int)
    ts = np.unique(P["prob"]); j = [((P["prob"] >= t) & (y == 1)).sum() / max(y.sum(), 1) + ((P["prob"] < t) & (y == 0)).sum() / max((y == 0).sum(), 1) for t in ts]
    return float(ts[int(np.argmax(j))])


sets = {"val": pd.read_pickle(run / "split_val.pkl"), "test": pd.read_pickle(run / "split_test.pkl")}
if a.ext:
    sets["external"] = D.load_dataset(a.ext, a.ext_root, expert=False)
    print(f"EXTERNAL set (evaluation only): {len(sets['external'])} samples")
preds = {k: predict(v) for k, v in sets.items()}
thr = youden(preds["val"], sets["val"]); print(f"decision threshold from validation: {thr:.3f}")
res = {}
for k, df in sets.items():
    if k == "val": continue
    c, per = clinical(preds[k]["texts"], df)
    res[k] = {"n": len(df), **image_level(preds[k], df, thr), "NLP": nlp(preds[k]["texts"], df), "clinical": c}
    if df.device.nunique() > 1:   # scanner-wise factual error rate
        res[k]["factual_error_by_device"] = {d: float(per[(df.device == d).values, 1].sum() / max(per[(df.device == d).values, 0].sum(), 1)) for d in df.device.unique()}
    pd.DataFrame({"generated": preds[k]["texts"], "reference": [D.build_report(x) for x in df.att]}).to_csv(run / f"reports_{k}.csv", index=False)
if "external" in res:  # Delta P = P_internal - P_external
    flat = lambda r: {**{m: v for m, v in r.items() if isinstance(v, float)}, **{m: v for m, v in r["NLP"].items() if isinstance(v, float)},
                      **{m: v for m, v in r["clinical"].items() if isinstance(v, float)}}
    fi, fe = flat(res["test"]), flat(res["external"])
    res["delta_internal_minus_external"] = {m: fi[m] - fe[m] for m in fi if m in fe}
json.dump(res, open(run / "metrics.json", "w"), indent=2)
print(json.dumps(res, indent=2))

## 3. Configuration
Set `EPOCHS` = `[S1, S2, S3, S4]`. Full run ≈ 1–2 h on a T4; use `[2,1,2,2]` for a quick check.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")             # results are copied to Drive so they survive a runtime reset
SAVE_DIR = "/content/drive/MyDrive/breast_vlreport_runs"
EPOCHS   = [15, 5, 15, 15]
BS       = 16
import os; os.makedirs(SAVE_DIR, exist_ok=True); os.makedirs("runs", exist_ok=True)

## 4. Data
**BUS-BRA** (development site) is downloaded from Zenodo (record 7730708).

In [ ]:
!wget -q -O BUSBRA.zip "https://zenodo.org/records/7730708/files/BUSBRA.zip?download=1"
!unzip -q -o BUSBRA.zip -d BUSBRA && ls BUSBRA | head
# If the download fails: download BUSBRA.zip manually from https://zenodo.org/record/7730708,
# upload it via the Colab file panel, and re-run the two lines above (without wget).

**BrEaST** (external site, CC-BY 4.0): get the dataset from the *Data availability* section of
Pawłowska et al., *Scientific Data* 11:148 (2024), doi:10.1038/s41597-024-02984-z (hosted on TCIA).
Put the downloaded folder/zip in your Google Drive, set the path below. It must contain the clinical `.xlsx`
plus the images and `*_tumor*` masks (searched recursively).

In [ ]:
BREAST_SRC = "/content/drive/MyDrive/BrEaST"      # <-- folder OR .zip in your Drive; edit this
import os, glob
if BREAST_SRC.endswith(".zip"):
    !unzip -q -o "$BREAST_SRC" -d BREAST
    BREAST = "BREAST"
else:
    BREAST = BREAST_SRC
print(len(glob.glob(BREAST + "/**/*.png", recursive=True)), "png files;", glob.glob(BREAST + "/**/*.xlsx", recursive=True))

## 5. Sanity check of the loaders (verify detected columns, counts, patient-level split)

In [ ]:
import data as D
a = D.load_dataset("busbra", "BUSBRA"); b = D.load_dataset("breast", BREAST)
print("BUS-BRA:", len(a), "samples,", a.group.nunique(), "patients | pathology:", a.att.map(lambda x: x["pathology"]).value_counts().to_dict())
print("BrEaST :", len(b), "samples | pathology:", b.att.map(lambda x: x["pathology"]).value_counts().to_dict())
print("scanners in BUS-BRA:", a.device.value_counts().to_dict())
print(D.build_report(a.att.iloc[0]))

## 6. Train the proposed model (4 stages)

In [ ]:
!python train.py --dev busbra --dev_root BUSBRA --model breast_vlreport --out runs/breast_vlreport \
    --epochs {" ".join(map(str,EPOCHS))} --bs {BS}

## 7. Evaluate: internal test + locked external test

In [ ]:
!python evaluate.py --run runs/breast_vlreport --ext breast --ext_root {BREAST}
!cp -r runs/breast_vlreport "{SAVE_DIR}/"

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 400)
r = pd.read_csv("runs/breast_vlreport/reports_external.csv")
for i in range(3): print("GENERATED:", r.generated[i], "\nREFERENCE:", r.reference[i], "\n")

## 8. Baselines (optional, ~same time each)

In [ ]:
for M in ["cnn_lstm", "cnn_transformer", "vit_transformer", "swin_transformer"]:
    !python train.py --dev busbra --dev_root BUSBRA --model {M} --out runs/{M} --epochs 0 0 {EPOCHS[2]*2} 0 --bs {BS}
    !python evaluate.py --run runs/{M} --ext breast --ext_root {BREAST}
    !cp -r runs/{M} "{SAVE_DIR}/"

## 9. Ablations (optional): G = direct image→report, H = +lesion, D = no contrastive

In [ ]:
for name, flag in [("abl_G","--no_lesion"), ("abl_H","--no_attr"), ("abl_D","--no_contrastive")]:
    !python train.py --dev busbra --dev_root BUSBRA --model breast_vlreport {flag} --out runs/{name} --epochs {" ".join(map(str,EPOCHS))} --bs {BS}
    !python evaluate.py --run runs/{name} --ext breast --ext_root {BREAST}
    !cp -r runs/{name} "{SAVE_DIR}/"

## 10. Comparison table (internal test vs external, with ΔP)

In [ ]:
import json, glob, pandas as pd
rows = []
for f in sorted(glob.glob("runs/*/metrics.json")):
    m, name = json.load(open(f)), f.split("/")[1]
    for split in ("test", "external"):
        if split in m:
            s = m[split]
            rows.append(dict(model=name, split=split, n=s["n"], **{k: s.get(k) for k in ["Dice","AUROC","birads_acc"]},
                             **{k: s["NLP"].get(k) for k in ["BLEU-4","ROUGE-L","CIDEr"]},
                             factual_err=s["clinical"]["factual_error_rate"], halluc_reports=s["clinical"]["hallucination_report_rate"],
                             omission=s["clinical"]["omission_rate"]))
df = pd.DataFrame(rows); display(df.round(3))
df.to_csv("runs/summary.csv", index=False); !cp runs/summary.csv "{SAVE_DIR}/"